### Compute power, sensitivity and PPV for results obtained over groups of datasets

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm

import folium
import branca.colormap as cm

In [2]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)
# display(dict_candidates)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

For each candidate, retrieve the grid and subset of cell it refers to.

In [3]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Retrieve the flattened list of object IDs associated with the candidates.
flattened_list_candidates = dict_candidates['flat_ids']

# Compute the starting pos, ending pos, and number of objects of each candidate.
start_pos_candidates = dict_candidates['start_pos'][:-1]
end_pos_candidates = dict_candidates['start_pos'][1:]
num_objs_candidates = np.diff(dict_candidates['start_pos'])

# Some DEBUG.
# print(len(start_pos_candidates), len(end_pos_candidates), len(num_objs_candidates))
# print(start_pos_candidates[-4], end_pos_candidates[-5])

In [4]:
# For each candidate, find out the resolution of the grid it comes from.
num_candidates = start_pos_candidates.size
candidates_grid_res = np.empty(num_candidates, dtype=np.uint32)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=np.uint32)
    grid_id[0] = grid[1]
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    candidates_grid_res[count : count + num_els_grid] = grid_id

    count += num_els_grid

# Create the list of grid resultions used during the assessment.
# NOTE: the zero resolution is used to consider the candidates from ALL the grids.
list_grid_res = [0] + np.unique(candidates_grid_res).tolist()


display(list_grid_res)
display(candidates_grid_res)

[0, 50, 75, 100, 200, 400, 600, 800, 1000]

array([ 600,  600,  600, ..., 1000, 1000, 1000],
      shape=(9696532,), dtype=uint32)

Now process the results...

In [5]:
name_set_groups_datasets = 'num_objects'
path_unfair_datasets = Path(f'./experiments/{name_set_groups_datasets}/')
list_files_results = sorted([f for f in path_unfair_datasets.iterdir() if f.is_file() and 'results' in f.name])
list_files_datasets = sorted([f for f in path_unfair_datasets.iterdir() if f.is_file() and 'results' not in f.name])
assert len(list_files_results) == len(list_files_datasets)


# Read the results computed over a given group of datasets from disk.
idx_tuple_list_objs = 1
dict_analyses = {}
for path_results, path_datasets in zip(list_files_results, list_files_datasets):
    print(f"Analyzing the results from {path_results} obtained from {path_datasets}")
    
    # Read the results computed over a group of datasets.
    with open(path_results, "rb") as f: set_results = pickle.load(f)
    # display(set_results)

    # Compute the number of datasets in this group.
    num_datasets_group = len(set_results['idx_candidates'])

    # Read the unfair datasets (needed to retrieve the list of object IDs belonging to the unfair hotspots).
    with open(path_datasets, "rb") as f:
        set_datasets = pickle.load(f)

    # Initialize the dictionary that will contain power, sensitivity, and PPV per subset of grids, and
    # for all the grids combined, for this group of datasets.
    # NOTE: resolution '0' means that we are considering all the subsets of grids.
    dict_analysis_res_group = {}
    for res in list_grid_res : dict_analysis_res_group[res] = \
        {'sum_power' : 0, 'sum_sensitivity' : 0., 'sum_ppv': 0.}


    # 2 - Compute power, sensitivity and PPV over this dataset.
    for idx_dataset in tqdm(range(num_datasets_group)) :

        # Retrieve the lists of object IDs associated with the various hotspots in this unfair dataset.
        # Each hotspot's list is a 1D numpy array, so we need to concatenate these arrays.
        # Guarantee also that the IDs in the final list are unique.
        set_unfair_obj_ids = np.unique(np.concatenate(set_datasets['data'][idx_dataset][idx_tuple_list_objs]))

        # Retrieve the candidates detected and thus deemed 'extreme' by the assessment approach
        # for this dataset.
        set_detected_candidates = set_results['idx_candidates'][idx_dataset]


        # 2.1 - Select the detected candidates that belong to a grid with resolution 'res'.
        for res in list_grid_res :
            
            # NOTE: if res == 0, then we consider all the candidates from all the grids.
            sel_set_detected_candidates = set_detected_candidates.copy()
            if res : sel_set_detected_candidates = \
                np.array([candidate for candidate in set_detected_candidates if candidates_grid_res[candidate] == res])

            # Retrieve the IDs of the objects associated with the extreme candidates found by the assessment
            # approach.
            list_detected_cand_objs = \
                [flattened_list_candidates[start_pos_candidates[i] : end_pos_candidates[i]] for i in sel_set_detected_candidates]
            list_detected_cand_objs = np.unique(np.concatenate(list_detected_cand_objs)) if list_detected_cand_objs else np.empty(0)

            # Compute the set intersection between the set of true unfair object IDs and the set of object IDs associated with
            # the candidates deemed 'extreme' by the assessment approach.
            intersect_obj_ids = np.intersect1d(set_unfair_obj_ids, list_detected_cand_objs)


            # Compute sensitivity and PPV. Recall:
            # - sensitivity:  measures the fraction of truly affected objects that are correctly flagged by an assessment approach
            # - ppv: measures the fraction of objects flagged by an assessment approach that truly belong to the true set.
            sensitivity = intersect_obj_ids.size / set_unfair_obj_ids.size
            ppv = intersect_obj_ids.size / list_detected_cand_objs.size if list_detected_cand_objs.size else 0.


            dict_analysis_res_group[res]['sum_power'] += (sel_set_detected_candidates.size != 0)
            dict_analysis_res_group[res]['sum_sensitivity'] += sensitivity
            dict_analysis_res_group[res]['sum_ppv'] += ppv


    # 3 - For each subset of grids of a given resolution, compute the statistical power, sensitivity, and PPV over the dataset group.
    print(f"DEBUG: Analysed group of datasets in {path_results.name}...")
    for res in list_grid_res :
        # Compute sensitivity and PPV for this resolution.
        #
        # NOTE: we skip the two divisions if dict_analysis_res_group[res]['sum_power'] == 0, because in these
        #       cases 'dict_analysis_res_group[res]['sum_sensitivity']' and 'dict_analysis_res_group[res]['sum_sensitivity']'
        #       are guaranteed to be already zero.
        if dict_analysis_res_group[res]['sum_power'] :
            dict_analysis_res_group[res]['sum_sensitivity'] = \
                dict_analysis_res_group[res]['sum_sensitivity'] / dict_analysis_res_group[res]['sum_power']
            dict_analysis_res_group[res]['sum_ppv'] = \
                dict_analysis_res_group[res]['sum_ppv'] / dict_analysis_res_group[res]['sum_power']

        # Compute the statistical power of the approach for this resolution.
        dict_analysis_res_group[res]['sum_power'] = \
            dict_analysis_res_group[res]['sum_power'] / num_datasets_group
        
        print(f"DEBUG: Power, sensitivity and PPV for grids with resolution {res}: {dict_analysis_res_group[res]['sum_power']}, " +
              f"{dict_analysis_res_group[res]['sum_sensitivity']}, " +
              f"{dict_analysis_res_group[res]['sum_ppv']}")
        

    # Add the analyses of the results from this file to the main dictionary.
    dict_analyses[path_results.name] = dict_analysis_res_group
    # break

Analyzing the results from experiments\num_objects\results_unfair_datasets_200_1_2_60_20.pkl obtained from experiments\num_objects\unfair_datasets_200_1_2_60_20.pkl


100%|██████████| 1000/1000 [00:01<00:00, 784.06it/s]


DEBUG: Analysed group of datasets in results_unfair_datasets_200_1_2_60_20.pkl...
DEBUG: Power, sensitivity and PPV for grids with resolution 0: 0.904, 0.7733604660487683, 0.38365547889559376
DEBUG: Power, sensitivity and PPV for grids with resolution 50: 0.198, 0.4345233889599496, 0.7207405556765941
DEBUG: Power, sensitivity and PPV for grids with resolution 75: 0.266, 0.43108642346847237, 0.7769398195291599
DEBUG: Power, sensitivity and PPV for grids with resolution 100: 0.337, 0.42121486369488187, 0.7938955791520377
DEBUG: Power, sensitivity and PPV for grids with resolution 200: 0.548, 0.452524076620597, 0.7867034639699961
DEBUG: Power, sensitivity and PPV for grids with resolution 400: 0.735, 0.5877161046450544, 0.6527601618192991
DEBUG: Power, sensitivity and PPV for grids with resolution 600: 0.705, 0.6867938083303898, 0.5117227554341517
DEBUG: Power, sensitivity and PPV for grids with resolution 800: 0.58, 0.7319886454291091, 0.4184147218062381
DEBUG: Power, sensitivity and PPV

100%|██████████| 1000/1000 [00:02<00:00, 360.93it/s]


DEBUG: Analysed group of datasets in results_unfair_datasets_300_1_2_60_20.pkl...
DEBUG: Power, sensitivity and PPV for grids with resolution 0: 0.986, 0.8659789445071867, 0.26230528583373497
DEBUG: Power, sensitivity and PPV for grids with resolution 50: 0.279, 0.45725341919511103, 0.6425160573001161
DEBUG: Power, sensitivity and PPV for grids with resolution 75: 0.364, 0.42451198513649796, 0.7037126870652471
DEBUG: Power, sensitivity and PPV for grids with resolution 100: 0.48, 0.39688251970807403, 0.7424325233792664
DEBUG: Power, sensitivity and PPV for grids with resolution 200: 0.741, 0.44334665331529843, 0.7522483960047004
DEBUG: Power, sensitivity and PPV for grids with resolution 400: 0.921, 0.618594480694022, 0.6091695855608947
DEBUG: Power, sensitivity and PPV for grids with resolution 600: 0.909, 0.726214191362793, 0.44745684188067136
DEBUG: Power, sensitivity and PPV for grids with resolution 800: 0.85, 0.7650977487744113, 0.34958984455461606
DEBUG: Power, sensitivity and P

...

DEBUG: Analysed group of datasets in results_unfair_datasets_400_1_2_60_20.pkl...
DEBUG: Power, sensitivity and PPV for grids with resolution 0: 0.998, 0.9256815130051677, 0.19297521617633392
DEBUG: Power, sensitivity and PPV for grids with resolution 50: 0.387, 0.5053219428182789, 0.5346683478506414
DEBUG: Power, sensitivity and PPV for grids with resolution 75: 0.503, 0.4617171669239544, 0.6129724882662644
DEBUG: Power, sensitivity and PPV for grids with resolution 100: 0.613, 0.43716233429994944, 0.6655963708277001
DEBUG: Power, sensitivity and PPV for grids with resolution 200: 0.852, 0.48686082945430154, 0.6867263446398323
DEBUG: Power, sensitivity and PPV for grids with resolution 400: 0.972, 0.6852779426315093, 0.5362228996817306
DEBUG: Power, sensitivity and PPV for grids with resolution 600: 0.966, 0.7936059707971908, 0.39106719710553917
DEBUG: Power, sensitivity and PPV for grids with resolution 800: 0.954, 0.8293803801803776, 0.29095770185209274
DEBUG: Power, sensitivity and

100%|██████████| 1000/1000 [00:07<00:00, 128.61it/s]


DEBUG: Analysed group of datasets in results_unfair_datasets_500_1_2_60_20.pkl...
DEBUG: Power, sensitivity and PPV for grids with resolution 0: 1.0, 0.9555402760163151, 0.1619718352012327
DEBUG: Power, sensitivity and PPV for grids with resolution 50: 0.44, 0.48856266048903063, 0.4933769266370885
DEBUG: Power, sensitivity and PPV for grids with resolution 75: 0.566, 0.4312688969940357, 0.5973152925069448
DEBUG: Power, sensitivity and PPV for grids with resolution 100: 0.681, 0.413167310606671, 0.644762190559581
DEBUG: Power, sensitivity and PPV for grids with resolution 200: 0.913, 0.48051274119166465, 0.665005128408763
DEBUG: Power, sensitivity and PPV for grids with resolution 400: 0.994, 0.7172236255947865, 0.4977944254702207
DEBUG: Power, sensitivity and PPV for grids with resolution 600: 0.993, 0.8420912849508327, 0.35072624107452594
DEBUG: Power, sensitivity and PPV for grids with resolution 800: 0.991, 0.8725810892796015, 0.25932480028353583
DEBUG: Power, sensitivity and PPV fo

100%|██████████| 1000/1000 [00:08<00:00, 116.75it/s]

DEBUG: Analysed group of datasets in results_unfair_datasets_600_1_2_60_20.pkl...
DEBUG: Power, sensitivity and PPV for grids with resolution 0: 1.0, 0.9753126207569299, 0.13525951713598616
DEBUG: Power, sensitivity and PPV for grids with resolution 50: 0.503, 0.506252147764662, 0.46416364714278746
DEBUG: Power, sensitivity and PPV for grids with resolution 75: 0.651, 0.4394498112330109, 0.5634473817446095
DEBUG: Power, sensitivity and PPV for grids with resolution 100: 0.769, 0.4244848260961748, 0.6043690530326591
DEBUG: Power, sensitivity and PPV for grids with resolution 200: 0.958, 0.5137039049911377, 0.6182373274971096
DEBUG: Power, sensitivity and PPV for grids with resolution 400: 1.0, 0.7670693465099516, 0.45978974412156765
DEBUG: Power, sensitivity and PPV for grids with resolution 600: 1.0, 0.8809332021833458, 0.3152901303109955
DEBUG: Power, sensitivity and PPV for grids with resolution 800: 0.999, 0.916214374907075, 0.2274613664332555
DEBUG: Power, sensitivity and PPV for g

In [ ]:
import pandas as pd
import re


dict_idx_parameter = {'num_objects' : 0, 'mag_unfair' : [3,4]}

# Put all the results gathered from a set of groups of dataset in a pandas dataframe.
final_df = []
idx_parameter = dict_idx_parameter[name_set_groups_datasets]
for k,v in dict_analyses.items() :
    params = re.findall(r'\d+', k)
    # print(params)
    dataframe_res = pd.DataFrame.from_dict(v).T

    if not isinstance(idx_parameter, list) :
        val_par = params[idx_parameter]  
    else :
        val_par = int(params[idx_parameter[0]]) - int(params[idx_parameter[1]]), \
                  int(params[idx_parameter[0]]), int(params[idx_parameter[1]])
    # print(val_par)
    dataframe_res['parameter'] = [val_par] * len(dataframe_res)
    # display(dataframe_res)
    
    final_df.append(dataframe_res)
final_df = pd.concat(final_df)
final_df.index.name = 'grid_res'
final_df.rename(index={0:'All'}, inplace=True)
# display(final_df)


# Now from the dataframe print the latex table to be copy/pasted in the paper.
grouped_df = final_df.groupby(final_df.index)
for key, orig_group in grouped_df:
    # Sort the rows within each group according to the values in the parameter column.
    item = orig_group.sort_values(by='parameter')
    # display(item)

    print("\\multirow{" + str(len(item)) + "}" + \
          "{*}{\\rotatebox[origin=c]{90}{\\textbf{" + \
          (str(key) if key else "All") + "}}}")
    
    for parameter, power, sensitivity, ppv in zip(item['parameter'], item['sum_power'], item['sum_sensitivity'], item['sum_ppv']) :
        # Case we are dealing with a 'normal' parameter...
        if not isinstance(parameter, tuple) :
            print("& $\\mathbf{" + parameter + "}$ " +  f"& {power:.3f} & {sensitivity:.3f} & {ppv:.3f} \\\\")
        # Case we are dealing with a pair of probabilities (unfairness magnitude)...
        else :
            pr_out = parameter[1] / 100
            pr_in = parameter[2] / 100
            delta = parameter[0] / 100
            print("& $\\mathbf{\Delta = " + f"{delta:.1f}" + "} \\ " + f"({pr_out:.1f} - {pr_in:.1f})" + "$ " + \
                   f"& {power:.3f} & {sensitivity:.3f} & {ppv:.3f} \\\\")
    print("\\midrule")
print("\\bottomrule")

\multirow{5}{*}{\rotatebox[origin=c]{90}{\textbf{50}}}
& $\mathbf{200}$ & 0.198 & 0.435 & 0.721 \\
& $\mathbf{300}$ & 0.279 & 0.457 & 0.643 \\
& $\mathbf{400}$ & 0.387 & 0.505 & 0.535 \\
& $\mathbf{500}$ & 0.440 & 0.489 & 0.493 \\
& $\mathbf{600}$ & 0.503 & 0.506 & 0.464 \\
\midrule
\multirow{5}{*}{\rotatebox[origin=c]{90}{\textbf{75}}}
& $\mathbf{200}$ & 0.266 & 0.431 & 0.777 \\
& $\mathbf{300}$ & 0.364 & 0.425 & 0.704 \\
& $\mathbf{400}$ & 0.503 & 0.462 & 0.613 \\
& $\mathbf{500}$ & 0.566 & 0.431 & 0.597 \\
& $\mathbf{600}$ & 0.651 & 0.439 & 0.563 \\
\midrule
\multirow{5}{*}{\rotatebox[origin=c]{90}{\textbf{100}}}
& $\mathbf{200}$ & 0.337 & 0.421 & 0.794 \\
& $\mathbf{300}$ & 0.480 & 0.397 & 0.742 \\
& $\mathbf{400}$ & 0.613 & 0.437 & 0.666 \\
& $\mathbf{500}$ & 0.681 & 0.413 & 0.645 \\
& $\mathbf{600}$ & 0.769 & 0.424 & 0.604 \\
\midrule
\multirow{5}{*}{\rotatebox[origin=c]{90}{\textbf{200}}}
& $\mathbf{200}$ & 0.548 & 0.453 & 0.787 \\
& $\mathbf{300}$ & 0.741 & 0.443 & 0.752 \\
& $